# 03. Advanced Interactive Visualization with `datamapplot`

This notebook demonstrates the final step in our analysis pipeline: creating a rich, interactive visualization of the NASA ADS dataset using `datamapplot`. We build upon the topic modeling results, incorporating advanced features like custom tooltips, an interactive legend, on-click actions, and a dynamic word cloud.

The goal is to produce a high-quality, explorable map that allows for deep insights into the thematic structure of the astrophysics literature.

## Imports

All required libraries are imported in a single, organized cell.

In [1]:
# Standard library imports
import os
import string
from pathlib import Path

# Third-party imports
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import rgb2hex
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Local and project-specific imports
import datamapplot
from datamapplot.selection_handlers import SelectionHandlerBase

## Setup

Define relative paths for data and plots to ensure portability and consistency.

In [2]:
import sys
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from config import DATA_DIR, PLOTS_DIR

data_path = DATA_DIR
plots_path = PLOTS_DIR

# Define the model used for topic labeling
model_name = 'text-embedding-3-large'
dim_reduction_name = 'pacmap_fast_hdbscan'
chosen_model = 'LiteLLM'

# Control whether the UI should render in dark mode when the plot is generated
ui_dark_mode = True

## Load and Prepare Data

Load the sampled dataset and prepare the necessary columns for visualization. This includes formatting text data, creating UMAP coordinates, and setting up metadata for tooltips and interactions.

In [3]:
df_sampled = pd.read_parquet(data_path / f"df_sampled_full_{model_name}_{dim_reduction_name}.parquet")

# Prepare data for datamapplot
data_map = df_sampled[['UMAP-1', 'UMAP-2']].to_numpy(np.float32)
label_layers = [df_sampled[chosen_model].to_numpy(object)]
hover_text = df_sampled.apply(lambda row: f"{row['Year']}", axis=1).tolist()

# Create a string version of tokens for the word cloud
df_sampled['tokens_str'] = df_sampled['tokens'].apply(' '.join)

# Prepare extra data for tooltips and interactions
extra_data = df_sampled[['Bibcode', 'Title_en', 'Author', 'Year', 'Journal', 'Abstract_en', 'Citation Count', 'DOI', 'tokens_str', 'Cluster', chosen_model]].copy()
extra_data.columns = ['bibcode', 'title', 'author', 'year', 'journal', 'abstract', 'citation_count', 'doi', 'tokens_str', 'cluster', 'primary_field']
extra_data['abstract'] = extra_data['abstract'].apply(lambda x: x[:200] + '...' if len(x) > 200 else x)
extra_data['citation_count'] = pd.to_numeric(extra_data['citation_count'], errors='coerce').fillna(0).astype(int)

# Prepare publication date for the histogram
df_sampled['publication_date'] = pd.to_datetime(df_sampled['Year'].astype(str) + '-12-31')

# Create color mapping for topics, with grey for noise
categories = extra_data['primary_field'].unique()
color_mapping = dict(zip(categories, map(rgb2hex, sns.color_palette("turbo", len(categories)))))
noise_labels = extra_data.loc[df_sampled['Cluster'] == -1, 'primary_field'].unique()
for label in noise_labels:
    color_mapping[label] = "#aaaaaa44"

extra_data["color"] = extra_data.primary_field.map(color_mapping)
marker_color_array = extra_data["color"]

# Set marker sizes based on citation count
marker_size_array = np.log1p(extra_data.citation_count.values) + 1

# Define histogram colors
palette = sns.color_palette("turbo", as_cmap=False)
bin_fill_color, selected_fill_color, unselected_fill_color, context_fill_color = map(rgb2hex, [palette[0], palette[5], palette[3], palette[1]])

## Define Interactive Components

This section contains all the custom code for enhancing the plot's interactivity. We define:
1.  **Custom CSS and HTML:** For a custom interactive legend.
2.  **Tooltip Template:** A custom HTML template for rich, informative tooltips.
3.  **Custom JavaScript:** To power the legend's click-to-select functionality.
4.  **WordCloud Class:** A `SelectionHandlerBase` subclass that displays a dynamic word cloud based on the user's selection.

In [4]:
# 1. Custom CSS for the interactive legend
custom_css = """
.row { display : flex; align-items : center; cursor: pointer; }
.row:hover { background-color: rgba(0,0,0,0.05); }
.box { height:10px; width:10px; border-radius:2px; margin-right:5px; padding:0px 0 1px 0; text-align:center; color: white; font-size: 14px; }
#legend { 
    position: absolute; 
    top: 0; 
    right: 0; 
    max-height: 80vh;
    display: flex;
    flex-direction: column;
}
#legend-header {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 5px 10px;
    border-bottom: 1px solid #ddd;
    cursor: pointer;
    font-weight: bold;
}
#legend-toggle { font-size: 18px; user-select: none; }
#legend-content {
    overflow-y: auto;
    max-height: calc(80vh - 40px);
    padding: 5px;
    transition: max-height 0.3s ease;
}
#legend-content.collapsed { max-height: 0; overflow: hidden; padding: 0 5px; }
#title-container { max-width: 75%; }

/* Tooltip komplett opaque - multiple selectors for datamapplot */
.tooltip, .hover-card, .hover-text-container, [class*="tooltip"], [class*="hover"] > div {
    background-color: #ffffff !important;
    background: #ffffff !important;
    opacity: 1 !important;
    backdrop-filter: blur(8px);
    box-shadow: 0 2px 8px rgba(0,0,0,0.2);
}
.container-box.more-opaque, .hover-card, .hover-text-container {
    background-color: #ffffff !important;
    background: #ffffff !important;
}

/* Dark mode styles - applied automatically when ui_dark_mode=True */
body.darkmode { background-color: #0d1117 !important; }
body.darkmode .container-box { background-color: rgba(22, 27, 34, 0.97) !important; color: #e0e0e0 !important; border-color: #30363d !important; }
body.darkmode .container-box.more-opaque { background-color: rgba(22, 27, 34, 0.97) !important; }
body.darkmode .tooltip, body.darkmode .hover-card, body.darkmode .hover-text-container, 
body.darkmode [class*="tooltip"], body.darkmode [class*="hover"] > div { 
    background-color: #161b22 !important; 
    background: #161b22 !important;
    color: #e0e0e0 !important; 
}
body.darkmode #legend-header { border-bottom-color: #30363d; }
body.darkmode .row:hover { background-color: rgba(255,255,255,0.1); }
body.darkmode #title-container { color: #e0e0e0; }
body.darkmode #title-container * { color: #e0e0e0 !important; }
body.darkmode .search-container input { background-color: #161b22; color: #e0e0e0; border-color: #30363d; }
body.darkmode .histogram-container, body.darkmode #word-cloud { background-color: rgba(22, 27, 34, 0.97) !important; }
"""

# 2. Custom HTML for the legend
custom_html = "<div id='legend' class='container-box'>\n"
custom_html += "    <div id='legend-header'><span>Topics</span><span id='legend-toggle'>▼</span></div>\n"
custom_html += "    <div id='legend-content'>\n"
for field, color in color_mapping.items():
    custom_html += f'        <div class="row"><div id="{field}" class="box" style="background-color:{color};"></div>{field}</div>\n'
custom_html += "    </div>\n"
custom_html += "</div>\n"

# 3. Custom tooltip template - mit explizitem weißen Hintergrund
badge_css = """
    border-radius:6px; width:fit-content; max-width:75%; margin:2px;
    padding: 2px 10px 2px 10px; font-size: 10pt;
"""
hover_text_template = f"""
<div style="background-color:#ffffff; padding:10px; border-radius:8px; box-shadow: 0 2px 12px rgba(0,0,0,0.25); min-width:250px;">
    <div style="font-size:12pt; font-weight:bold; padding:2px; color:#111;">{{title}}</div>
    <div style="color:#333;"><b>Author:</b> {{author}}</div>
    <div style="color:#333;"><b>Year:</b> {{year}}</div>
    <div style="color:#333;"><b>Journal:</b> {{journal}}</div>
    <div style="color:#333;"><b>Abstract:</b> {{abstract}}</div>
    <div style="background-color:{{color}}; color:#fff; {badge_css}">{{primary_field}}</div>
    <div style="background-color:#eeeeee; color:#333; {badge_css}">citation count: {{citation_count}}</div>
</div>
"""

# 4. Custom JavaScript for legend interactivity and dark mode
apply_dark_ui_js = str(ui_dark_mode).lower()
custom_js = """
const APPLY_DARK_UI = __APPLY_DARK_UI__;
if (APPLY_DARK_UI) {
    document.body.classList.add('darkmode');
}

const legend = document.getElementById("legend");
const legendContent = document.getElementById("legend-content");
const legendToggle = document.getElementById("legend-toggle");
const legendHeader = document.getElementById("legend-header");
const selectedPrimaryFields = new Set();

// Toggle Legend collapse/expand
legendHeader.addEventListener('click', function(event) {
    if (event.target.closest('.box')) return;
    legendContent.classList.toggle('collapsed');
    legendToggle.textContent = legendContent.classList.contains('collapsed') ? '▶' : '▼';
});

// Legend click-to-select functionality
legendContent.addEventListener('click', function (event) {
    const row = event.target.closest('.row');
    if (!row) return;
    const box = row.querySelector('.box');
    const selectedField = box ? box.id : null;
    if (selectedField) {
        if (selectedPrimaryFields.has(selectedField)) {
            selectedPrimaryFields.delete(selectedField);
            box.innerHTML = "";
        } else {
            selectedPrimaryFields.add(selectedField);
            box.innerHTML = "✓";
        }
    }
    const selectedIndices = [];
    datamap.metaData.primary_field.forEach((field, i) => {
        if (selectedPrimaryFields.has(field)) selectedIndices.push(i);
    });
    datamap.addSelection(selectedIndices, "legend");
});

// Reduce Y-axis ticks on histogram - show only every 3rd tick
setTimeout(() => {
    const svgElements = document.querySelectorAll('svg');
    svgElements.forEach(svg => {
        const ticks = svg.querySelectorAll('.tick');
        if (ticks.length > 5) {
            ticks.forEach((tick, i) => { 
                if (i % 3 !== 0) tick.style.opacity = '0'; 
            });
        }
    });
}, 2000);
"""
custom_js = custom_js.replace("__APPLY_DARK_UI__", apply_dark_ui_js)

# 5. WordCloud Selection Handler Class
class WordCloud(SelectionHandlerBase):
    def __init__(
        self,
        n_words=256,
        width=500,
        height=500,
        font_family=None,
        stop_words=None,
        n_rotations=0,
        color_scale="YlGnBu",
        location=("bottom", "right"),
        text_field='tokens_str',
        **kwargs,
    ):
        super().__init__(
            dependencies=[
                "https://d3js.org/d3.v6.min.js",
                "https://unpkg.com/d3-cloud@1.2.7/build/d3.layout.cloud.js",
                "https://ajax.googleapis.com/ajax/libs/jquery/3.7.1/jquery.min.js",
            ],
            **kwargs,
        )
        self.n_words = n_words
        self.width = width
        self.height = height
        self.font_family = font_family
        self.stop_words = stop_words or list(ENGLISH_STOP_WORDS)
        self.n_rotations = min(22, n_rotations)
        self.location = location
        self.text_field = text_field
        if color_scale.endswith("_r"):
            self.color_scale = string.capwords(color_scale[:1]) + color_scale[1:-2]
            self.color_scale_reversed = True
        else:
            self.color_scale = string.capwords(color_scale[:1]) + color_scale[1:]
            self.color_scale_reversed = False

    @property
    def javascript(self):
        return f"""
const _STOPWORDS = new Set({self.stop_words});
const _ROTATIONS = [0, -90, 90, -45, 45, -30, 30, -60, 60, -15, 15, -75, 75, -7.5, 7.5, -22.5, 22.5, -52.5, 52.5, -37.5, 37.5, -67.5, 67.5];
const wordCloudSvg = d3.select("#word-cloud").append("svg")
    .attr("width", {self.width})
    .attr("height", {self.height})
    .append("g")
    .attr("transform", "translate(" + {self.width} / 2 + "," + {self.height} / 2 + ")");
const wordCloudItem = document.getElementById("word-cloud");

function wordCounter(textItems) {{
    const words = textItems.join(' ').toLowerCase().split(/\s+/);
    const wordCounts = new Map();
    words.forEach(word => {{
        wordCounts.set(word, (wordCounts.get(word) || 0) + 1);
    }});
    _STOPWORDS.forEach(stopword => wordCounts.delete(stopword));
    const result = Array.from(wordCounts, ([word, frequency]) => ({{ text: word, size: Math.sqrt(frequency) }}))
                         .sort((a, b) => b.size - a.size).slice(0, {self.n_words});
    const maxSize = Math.max(...(result.map(x => x.size)));
    return result.map(({{text, size}}) => ({{ text: text, size: (size / maxSize)}}));
}}

function generateWordCloud(words) {{
  const width = {self.width};
  const height = {self.height};
  const colorScale = d3.scaleSequential(d3.interpolate{self.color_scale}).domain([{"width / 10, 0" if self.color_scale_reversed else "0, width / 10"}]);
  const layout = d3.layout.cloud()
    .size([width, height])
    .words(words.map(d => ({{text: d.text, size: d.size * width / 10}})))
    .padding(1)
    .rotate(() => _ROTATIONS[~~(Math.random() * {self.n_rotations})])
    .font("{self.font_family or 'Impact'}")
    .fontSize(d => d.size)
    .fontWeight(d => Math.max(300, Math.min(d.size * 9000 / width, 900)))
    .on("end", draw);
  layout.start();

  function draw(words) {{
    const t = d3.transition().duration(500);
    const text = wordCloudSvg.selectAll("text")
      .data(words, d => d.text);
    text.exit()
      .transition(t)
      .attr("fill-opacity", 0)
      .attr("font-size", 1)
      .remove();
    text.enter()
      .append("text")
      .attr("text-anchor", "middle")
      .attr("fill-opacity", 0)
      .attr("font-size", 1)
      .attr("font-family", "{self.font_family or 'Impact'}")
      .text(d => d.text)
      .merge(text)
      .transition(t)
      .attr("transform", d => "translate(" + [d.x, d.y] + ")rotate(" + d.rotate + ")")
      .attr("fill-opacity", 1)
      .attr("font-size", d => d.size)
      .attr("font-weight", d => Math.max(300, Math.min(d.size * 9000 / width, 900)))
      .attr("fill", d => colorScale(d.size));
  }}
}}

function lassoSelectionCallback(selectedPoints) {{
    if (selectedPoints.length > 0) {{
        $(wordCloudItem).animate({{height:'show'}}, 250);
    }} else {{
        $(wordCloudItem).animate({{height:'hide'}}, 250);
        return;
    }}
    let selectedText;
    if (datamap.metaData && datamap.metaData.{self.text_field}) {{
        selectedText = selectedPoints.map(i => datamap.metaData.{self.text_field}[i]);
    }} else {{
        selectedText = ["Meta data still loading ..."];
    }}
    const wordCounts = wordCounter(selectedText);
    generateWordCloud(wordCounts);
}}
"""

    @property
    def html(self):
        return '''<div id="word-cloud" class="container-box more-opaque"></div>'''

    @property
    def css(self):
        return f"""
#word-cloud {{
    position: absolute;
    {self.location[1]}: 0;
    {self.location[0]}: 0;
    display: none;
    width: {self.width}px;
    height: {self.height}px;
    z-index: 10;
}}
"""

<>:269: SyntaxWarning: invalid escape sequence '\s'
<>:269: SyntaxWarning: invalid escape sequence '\s'
C:\Users\rapha\AppData\Local\Temp\ipykernel_17976\4163847777.py:269: SyntaxWarning: invalid escape sequence '\s'
  


## Generate Plot

With all data and components prepared, we now generate the final interactive plot. All parameters from the debug notebook are preserved to ensure feature parity. The plot is displayed in the notebook and saved to the `plots` directory.

In [5]:
plot = datamapplot.create_interactive_plot(
    # Core Data and Configuration
    data_map,                      # 2D coordinates for data points
    *label_layers,                 # Label layers for cluster resolutions
    hover_text=hover_text,         # Tooltip text for hover effects
    extra_point_data=extra_data,   # Additional data per point
    inline_data=True,              # Embed data inline in the HTML

    # Plot Appearance Settings
    title="NASA ADS - BERTopic",
    sub_title=f"Topics labeled with {chosen_model}",
    font_family="Cinzel",
    enable_search=True,
    search_field="author",
    initial_zoom_fraction=0.99,
    darkmode=ui_dark_mode,
    cluster_boundary_polygons=True,
    cluster_boundary_line_width=8,
    use_medoids=True,

    # Marker and Styling Options
    marker_size_array=marker_size_array,
    marker_color_array=marker_color_array,
    point_radius_max_pixels=20,
    point_radius_min_pixels=2,
    noise_label=', '.join(noise_labels),
    color_label_text=False,
    color_cluster_boundaries=False,

    # Text and Label Settings
    text_outline_width=4,
    text_min_pixel_size=16,
    text_max_pixel_size=48,
    min_fontsize=16,
    max_fontsize=32,

    # Time-Series Histogram Settings
    histogram_data=df_sampled['publication_date'],
    histogram_group_datetime_by="year",
    histogram_range=(pd.to_datetime("1911-01-01"), pd.to_datetime("2001-01-01")),
    histogram_settings={
        "histogram_log_scale": True,
        "histogram_title": "Publications per Year",
        "histogram_width": 400,
        "histogram_height": 150,
        "histogram_bin_fill_color": bin_fill_color,
        "histogram_bin_selected_fill_color": selected_fill_color,
        "histogram_bin_unselected_fill_color": unselected_fill_color,
        "histogram_bin_context_fill_color": context_fill_color,
    },

    # Custom Hover Text and On-Click Action
    hover_text_html_template=hover_text_template,
    on_click="window.open(`https://ui.adsabs.harvard.edu/abs/{bibcode}/abstract`)",

    # Custom Interactive Components
    custom_css=custom_css,
    custom_html=custom_html,
    custom_js=custom_js,

    # WordCloud Selection Handler
    selection_handler=WordCloud(
        n_words=256,
        width=500,
        height=300,
        color_scale="turbo",
        font_family="Cinzel",
        text_field='tokens_str'
    )
)

# Save the plot to the plots directory
plot_filename = f"ADS_BERTopic-Map-{model_name}.html"
plot.save(plots_path / plot_filename)

# Display the plot in the notebook
# plot